# Personalized Healthcare Recommendation ML Model

This notebook generates a synthetic dataset based on health and lifestyle metrics, trains a Decision Tree classifier, and exports it to JavaScript!

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import m2cgen as m2c
import json

print("Generating synthetic personalized healthcare dataset...")

np.random.seed(42)
n_samples = 2000

ages = np.random.randint(18, 80, n_samples)
bmis = np.random.uniform(16.0, 40.0, n_samples)
sleep_hours = np.random.uniform(4.0, 10.0, n_samples)
exercise_hours = np.random.uniform(0.0, 14.0, n_samples)
stress_levels = np.random.randint(1, 11, n_samples)

# Simple rules for recommendations
classes = []
for i in range(n_samples):
    if bmis[i] > 30 and exercise_hours[i] < 3:
        classes.append('Weight Management & Cardio Plan')
    elif stress_levels[i] > 7 and sleep_hours[i] < 6:
        classes.append('Stress Reduction & Sleep Therapy Plan')
    elif exercise_hours[i] > 7 and bmis[i] < 25:
        classes.append('High Performance & Nutrition Maintenance')
    elif ages[i] > 60:
        classes.append('Senior Wellness & Joint Care Plan')
    else:
        classes.append('General Balanced Health Routine')

df = pd.DataFrame({
    'Age': ages,
    'BMI': bmis,
    'SleepHours': sleep_hours,
    'ExerciseHours': exercise_hours,
    'StressLevel': stress_levels
})
y = np.array(classes)

# We map classes to integers so m2cgen handles it nicely
class_mapping = {label: idx for idx, label in enumerate(np.unique(y))}
idx_to_class = {idx: label for label, idx in class_mapping.items()}
y_encoded = np.array([class_mapping[label] for label in y])

print("Training Decision Tree Model...")
# Train model using standard numerical outputs
model = DecisionTreeClassifier(max_depth=5)
model.fit(df, y_encoded)

print("Exporting model to JavaScript...")
# Convert to JS
js_code = m2c.export_to_javascript(model)

# The m2cgen library exports a JS function named `score(input)`
# We will wrap it and export it nicely so our React app can use it
js_wrapper = f"""
// Auto-generated Machine Learning Model
// Predicts Health Recommendation based on: [Age, BMI, SleepHours, ExerciseHours, StressLevel]

const CLASS_MAPPING = {json.dumps(idx_to_class, indent=2)};

{js_code}

export function predictRecommendation(age, bmi, sleepHours, exerciseHours, stressLevel) {{
  const input = [age, bmi, sleepHours, exerciseHours, stressLevel];
  
  // score is generated by m2cgen
  const outputScores = score(input);
  
  // The output is an array of probabilities/scores for each class
  // Find the index of the max score
  let maxIdx = 0;
  let maxScore = outputScores[0];
  for (let i = 1; i < outputScores.length; i++) {{
    if (outputScores[i] > maxScore) {{
      maxScore = outputScores[i];
      maxIdx = i;
    }}
  }}
  
  return CLASS_MAPPING[maxIdx];
}}
"""

with open('healthcare-app/src/recommendation_model.js', 'w') as f:
    f.write(js_wrapper)

print("saving dataset as CSV for reference...")
df['Recommendation'] = y
df.to_csv('synthetic_healthcare_data.csv', index=False)

print("Model exported successfully to healthcare-app/src/recommendation_model.js")
